# GeoQuery Results — {{REQUEST_NAME}}

**Request ID:** `{{REQUEST_ID}}`  
**Generated:** {{DATE}}

This notebook downloads your GeoQuery results and walks through loading,
mapping, and summarizing the data. Run the cells in order.

## 1. Setup

In [ ]:
%pip install geopandas matplotlib -q

In [ ]:
import pathlib
import urllib.request
import zipfile

DOWNLOAD_URL = "{{DOWNLOAD_URL}}"
OUT = pathlib.Path("geoquery_results")
OUT.mkdir(exist_ok=True)

zip_path = OUT / "results.zip"
print("Downloading results…")
urllib.request.urlretrieve(DOWNLOAD_URL, zip_path)

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(OUT)

files = sorted(f for f in OUT.rglob("*") if f.is_file() and f.suffix != ".zip")
print(f"Extracted {len(files)} file(s):")
for f in files:
    print(f"  {f}")

## 2. Load tabular results (CSV)

In [ ]:
import pandas as pd

csv_files = sorted(OUT.rglob("*.csv"))
df = pd.read_csv(csv_files[0])
print(f"{len(df):,} rows · {len(df.columns)} columns · {csv_files[0].name}")
df.head()

In [ ]:
df.describe()

## 3. Load spatial data (GeoPackage)

In [ ]:
import geopandas as gpd

gpkg_files = sorted(OUT.rglob("*.gpkg"))
gdf = gpd.read_file(gpkg_files[0])
print(f"{len(gdf):,} features · {gpkg_files[0].name}")
gdf.head()

## 4. Join CSV to spatial data

In [ ]:
# The CSV and GeoPackage share a common 'geom_id' column.
# This merge lets you map any CSV column spatially.
joined = gdf[["geom_id", "geometry"]].merge(df, on="geom_id", how="left")
joined.head()

## 5. Choropleth map

In [ ]:
import matplotlib.pyplot as plt

# Change `col` to any extract column from df.columns
extract_cols = [c for c in df.columns if c not in ["geom_id", "feature_collection"] and not c.startswith("boundary.")]
col = extract_cols[0] if extract_cols else None

fig, ax = plt.subplots(figsize=(12, 7))
joined.plot(column=col, legend=True, cmap="YlOrRd", ax=ax)
ax.set_title(col or "Features")
ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Time series (mean per extract column)

In [ ]:
# Mean of each extract column — useful when columns represent time periods.
extract_cols = [c for c in df.columns if c not in ["geom_id", "feature_collection"] and not c.startswith("boundary.")]
extract_numeric = df[extract_cols].select_dtypes("number")
means = extract_numeric.mean()

fig, ax = plt.subplots(figsize=(12, 4))
means.plot(ax=ax, marker="o")
ax.set_title("Mean value per extract column")
ax.set_ylabel("Mean")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()